In [15]:
# Créer la session Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence")
    .master("local[*]")
    .getOrCreate()
)

print("Version Spark :", spark.version)
print("Mode d'exécution :", spark.sparkContext.master)
print("Cœurs utilisés :", spark.sparkContext.defaultParallelism)

Version Spark : 4.3.0
Mode d'exécution : local[*]
Cœurs utilisés : 2


In [16]:
# Lire un fichier de test depuis HDFS
chemin_entree = "hdfs:///retail-lake/raw/tpcxbb/initial/item/item.dat"

df = (
    spark.read
    .option("header", False)
    .option("inferSchema", True)
    .option("sep", "|")
    .csv(chemin_entree)
)

df.show(5)
print("Nombre de lignes :", df.count())

+---+----------------+----------+----+--------------------+-----+-----+-------+--------------------+---+--------------------+----+-----------------+----+--------------------+------+--------------------+--------+-----+-------+----+--------------------+
|_c0|             _c1|       _c2| _c3|                 _c4|  _c5|  _c6|    _c7|                 _c8|_c9|                _c10|_c11|             _c12|_c13|                _c14|  _c15|                _c16|    _c17| _c18|   _c19|_c20|                _c21|
+---+----------------+----------+----+--------------------+-----+-----+-------+--------------------+---+--------------------+----+-----------------+----+--------------------+------+--------------------+--------+-----+-------+----+--------------------+
|  0|AAAAAAAAAAAAAAAA|2000-01-14|NULL|quickly even dino...|72.29|64.96|3898712|71TS7NSbvH1YbdiQM...|  1|            Fan Shop|   9|Sports & Outdoors| 995|    2VOxvrIWwlJQTSk6| small|  99Ee1r6uFbZSSClAX3|  dodger|   Oz|Unknown|  18|8m9n5Q7T33DNWi

In [17]:
# Afficher le schéma
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: date (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: double (nullable = true)
 |-- _c6: double (nullable = true)
 |-- _c7: integer (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: integer (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: integer (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: integer (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)
 |-- _c20: integer (nullable = true)
 |-- _c21: string (nullable = true)



In [19]:
# écrire un résultat de test dans un dossier autorisé, puis le relire
chemin_sortie = "file:///home/noah/retail-intelligence/tp3/outputs/test_ecriture"

df.limit(10).write.mode("overwrite").csv(chemin_sortie, header=True)

df_relu = spark.read.option("header", True).csv(chemin_sortie)
df_relu.show()
print("Lignes relues :", df_relu.count())

+---+----------------+----------+----+--------------------+-----+-----+-------+--------------------+---+--------------------+----+-----------------+----+--------------------+-------+--------------------+--------+-----+-------+----+--------------------+
|_c0|             _c1|       _c2| _c3|                 _c4|  _c5|  _c6|    _c7|                 _c8|_c9|                _c10|_c11|             _c12|_c13|                _c14|   _c15|                _c16|    _c17| _c18|   _c19|_c20|                _c21|
+---+----------------+----------+----+--------------------+-----+-----+-------+--------------------+---+--------------------+----+-----------------+----+--------------------+-------+--------------------+--------+-----+-------+----+--------------------+
|  0|AAAAAAAAAAAAAAAA|2000-01-14|NULL|quickly even dino...|72.29|64.96|3898712|71TS7NSbvH1YbdiQM...|  1|            Fan Shop|   9|Sports & Outdoors| 995|    2VOxvrIWwlJQTSk6|  small|  99Ee1r6uFbZSSClAX3|  dodger|   Oz|Unknown|  18|8m9n5Q7T33

In [21]:
# Lire les deux sources avec les vrais noms de colonnes
from pyspark.sql import functions as F

colonnes_store_sales = [
    "ss_sold_date_sk", "ss_sold_time_sk", "ss_item_sk", "ss_customer_sk", "ss_cdemo_sk",
    "ss_hdemo_sk", "ss_addr_sk", "ss_store_sk", "ss_promo_sk", "ss_ticket_number",
    "ss_quantity", "ss_wholesale_cost", "ss_list_price", "ss_sales_price",
    "ss_ext_discount_amt", "ss_ext_sales_price", "ss_ext_wholesale_cost",
    "ss_ext_list_price", "ss_ext_tax", "ss_coupon_amt", "ss_net_paid",
    "ss_net_paid_inc_tax", "ss_net_profit"
]

colonnes_item = [
    "i_item_sk", "i_item_id", "i_rec_start_date", "i_rec_end_date", "i_item_desc",
    "i_current_price", "i_wholesale_cost", "i_brand_id", "i_brand", "i_class_id",
    "i_class", "i_category_id", "i_category", "i_manufact_id", "i_manufact",
    "i_size", "i_formulation", "i_color", "i_units", "i_container",
    "i_manager_id", "i_product_name"
]

def lire_source(chemin, colonnes):
    df = (
        spark.read
        .option("header", False)
        .option("inferSchema", True)
        .option("sep", "|")
        .csv(chemin)
    )
    print("Colonnes lues :", len(df.columns), "/ attendues :", len(colonnes))
    return df.toDF(*colonnes)

df_ventes = lire_source("hdfs:///retail-lake/raw/tpcxbb/initial/store_sales/store_sales.dat", colonnes_store_sales)
df_produits = lire_source("hdfs:///retail-lake/raw/tpcxbb/initial/item/item.dat", colonnes_item)

Colonnes lues : 23 / attendues : 23
Colonnes lues : 22 / attendues : 22


In [22]:
# Schéma, échantillon et nombre de lignes
for nom, df in [("store_sales", df_ventes), ("item", df_produits)]:
    print("=====", nom)
    df.printSchema()
    df.show(3, truncate=True)
    print("Lignes :", df.count(), "| Colonnes :", len(df.columns))

===== store_sales
root
 |-- ss_sold_date_sk: integer (nullable = true)
 |-- ss_sold_time_sk: integer (nullable = true)
 |-- ss_item_sk: integer (nullable = true)
 |-- ss_customer_sk: integer (nullable = true)
 |-- ss_cdemo_sk: integer (nullable = true)
 |-- ss_hdemo_sk: integer (nullable = true)
 |-- ss_addr_sk: integer (nullable = true)
 |-- ss_store_sk: integer (nullable = true)
 |-- ss_promo_sk: integer (nullable = true)
 |-- ss_ticket_number: integer (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: double (nullable = true)
 |-- ss_list_price: double (nullable = true)
 |-- ss_sales_price: double (nullable = true)
 |-- ss_ext_discount_amt: double (nullable = true)
 |-- ss_ext_sales_price: double (nullable = true)
 |-- ss_ext_wholesale_cost: double (nullable = true)
 |-- ss_ext_list_price: double (nullable = true)
 |-- ss_ext_tax: double (nullable = true)
 |-- ss_coupon_amt: double (nullable = true)
 |-- ss_net_paid: double (nullable = true)
 |-- s

Lignes : 667579 | Colonnes : 23
===== item
root
 |-- i_item_sk: integer (nullable = true)
 |-- i_item_id: string (nullable = true)
 |-- i_rec_start_date: date (nullable = true)
 |-- i_rec_end_date: string (nullable = true)
 |-- i_item_desc: string (nullable = true)
 |-- i_current_price: double (nullable = true)
 |-- i_wholesale_cost: double (nullable = true)
 |-- i_brand_id: integer (nullable = true)
 |-- i_brand: string (nullable = true)
 |-- i_class_id: integer (nullable = true)
 |-- i_class: string (nullable = true)
 |-- i_category_id: integer (nullable = true)
 |-- i_category: string (nullable = true)
 |-- i_manufact_id: integer (nullable = true)
 |-- i_manufact: string (nullable = true)
 |-- i_size: string (nullable = true)
 |-- i_formulation: string (nullable = true)
 |-- i_color: string (nullable = true)
 |-- i_units: string (nullable = true)
 |-- i_container: string (nullable = true)
 |-- i_manager_id: integer (nullable = true)
 |-- i_product_name: string (nullable = true)

+--

In [23]:
# Colonnes qui contiennent des valeurs nulles
for nom, df in [("store_sales", df_ventes), ("item", df_produits)]:
    nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).first().asDict()
    print("=====", nom)
    for colonne, nb in nulls.items():
        if nb > 0:
            print(colonne, ":", nb)

===== store_sales
ss_cdemo_sk : 238
ss_hdemo_sk : 295
ss_addr_sk : 448
ss_promo_sk : 158


[Stage 35:=============================>                            (1 + 1) / 2]

===== item
i_rec_end_date : 17820
i_item_desc : 4
i_brand_id : 8
i_brand : 6


In [24]:
# Les identifiants sont-ils uniques ? (clé candidate)
print("item - lignes :", df_produits.count())
print("item - i_item_sk distincts :", df_produits.select("i_item_sk").distinct().count())
print("item - i_item_id distincts :", df_produits.select("i_item_id").distinct().count())

print("store_sales - lignes :", df_ventes.count())
print("store_sales - (ticket, produit) distincts :",
      df_ventes.select("ss_ticket_number", "ss_item_sk").distinct().count())

item - lignes : 17820


item - i_item_sk distincts : 17820
item - i_item_id distincts : 17820
store_sales - lignes : 667579


[Stage 58:>                                                         (0 + 2) / 2]

store_sales - (ticket, produit) distincts : 663443


In [25]:
# Valeurs minimum et maximum de chaque colonne numérique
def min_max(df, nom):
    colonnes_num = [c for c, t in df.dtypes if t in ("int", "double")]
    resultat = df.agg(
        *[F.min(c).alias(c + "_min") for c in colonnes_num],
        *[F.max(c).alias(c + "_max") for c in colonnes_num]
    ).first()
    print("=====", nom)
    for c in colonnes_num:
        print(f"{c:25} min = {resultat[c + '_min']:>12}   max = {resultat[c + '_max']:>12}")

min_max(df_ventes, "store_sales")
min_max(df_produits, "item")

26/11/18 14:22:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

===== store_sales
ss_sold_date_sk           min =        36890   max =        38697
ss_sold_time_sk           min =            0   max =        86399
ss_item_sk                min =            0   max =        17819
ss_customer_sk            min =            2   max =        98999
ss_cdemo_sk               min =           18   max =      1920742
ss_hdemo_sk               min =            0   max =         7199
ss_addr_sk                min =            0   max =        49499
ss_store_sk               min =            0   max =           11
ss_promo_sk               min =            0   max =          299
ss_ticket_number          min =            0   max =        89099
ss_quantity               min =            1   max =          100
ss_wholesale_cost         min =          1.0   max =        100.0
ss_list_price             min =          1.0   max =        200.0
ss_sales_price            min =          0.0   max =        199.3
ss_ext_discount_amt       min =          0.0   max =      

In [26]:
# Doublons exacts (lignes identiques sur toutes les colonnes)
for nom, df in [("store_sales", df_ventes), ("item", df_produits)]:
    total = df.count()
    sans_doublons = df.dropDuplicates().count()
    print(nom, "- lignes :", total, "| lignes différentes :", sans_doublons,
          "| doublons exacts :", total - sans_doublons)

store_sales - lignes : 667579 | lignes différentes : 667579 | doublons exacts : 0


[Stage 80:=============================>                            (1 + 1) / 2]

item - lignes : 17820 | lignes différentes : 17820 | doublons exacts : 0


In [27]:
# Regarder les 4 136 couples (ticket, produit) répétés
couples_repetes = (
    df_ventes.groupBy("ss_ticket_number", "ss_item_sk")
    .count()
    .filter(F.col("count") > 1)
)
print("Couples répétés :", couples_repetes.count())
couples_repetes.groupBy("count").count().withColumnRenamed("count", "nb_couples").show()

# Exemple : les lignes complètes d'un couple répété
exemple = couples_repetes.first()
df_ventes.filter(
    (F.col("ss_ticket_number") == exemple["ss_ticket_number"]) &
    (F.col("ss_item_sk") == exemple["ss_item_sk"])
).show(truncate=False)

Couples répétés : 4038


+----------+----------+
|nb_couples|nb_couples|
+----------+----------+
|         3|        94|
|         2|      3942|
|         4|         2|
+----------+----------+



[Stage 102:>                                                        (0 + 1) / 1]

+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+

In [28]:
# Valeurs atypiques ou incohérentes dans store_sales
controles_ventes = {
    "quantité <= 0": F.col("ss_quantity") <= 0,
    "prix de vente <= 0": F.col("ss_sales_price") <= 0,
    "prix de vente > prix catalogue": F.col("ss_sales_price") > F.col("ss_list_price"),
    "montant payé < 0": F.col("ss_net_paid") < 0,
    "marge < 0": F.col("ss_net_profit") < 0,
    "montant ≠ quantité × prix (écart > 0,01)":
        F.abs(F.col("ss_ext_sales_price") - F.col("ss_quantity") * F.col("ss_sales_price")) > 0.01,
}
for libelle, condition in controles_ventes.items():
    print(f"{libelle:45} : {df_ventes.filter(condition).count()}")

quantité <= 0                                 : 0


prix de vente <= 0                            : 3324


prix de vente > prix catalogue                : 0


montant payé < 0                              : 0


marge < 0                                     : 495971


[Stage 118:>                                                        (0 + 2) / 2]

montant ≠ quantité × prix (écart > 0,01)      : 0


In [29]:
# Valeurs atypiques ou incohérentes dans item
# Catégories présentes
df_produits.groupBy("i_category").count().orderBy("count", ascending=False).show(truncate=False)

# Valeurs "vides déguisées" dans les colonnes texte
colonnes_texte = [c for c, t in df_produits.dtypes if t == "string"]
for c in colonnes_texte:
    nb = df_produits.filter(F.trim(F.col(c)).isin("", "N/A", "Unknown", "NO_SUB_CAT")).count()
    if nb > 0:
        print(f"{c:20} : {nb} valeurs vides / N/A / Unknown / NO_SUB_CAT")

# Prix
print("prix <= 0 :", df_produits.filter(F.col("i_current_price") <= 0).count())
print("coût d'achat > prix de vente :",
      df_produits.filter(F.col("i_wholesale_cost") > F.col("i_current_price")).count())

+------------------------+-----+
|i_category              |count|
+------------------------+-----+
|Music                   |8143 |
|Books                   |5335 |
|Movies & TV             |1286 |
|Electronics             |692  |
|Clothing & Accessories  |610  |
|Home & Kitchen          |476  |
|Sports & Outdoors       |465  |
|Tools & Home Improvement|408  |
|Toys & Games            |405  |
+------------------------+-----+

i_class              : 1576 valeurs vides / N/A / Unknown / NO_SUB_CAT
i_size               : 2502 valeurs vides / N/A / Unknown / NO_SUB_CAT
i_units              : 1767 valeurs vides / N/A / Unknown / NO_SUB_CAT
i_container          : 17820 valeurs vides / N/A / Unknown / NO_SUB_CAT
prix <= 0 : 0
coût d'achat > prix de vente : 7768
